```sql
05 - Velocity Features (Past-only)

Goal:
- add a first set of velocity features (counts / sums in recent windows)
- keep it leakage-safe (past-only)
- compare against the same validation split

```

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

#### Import Data

In [3]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [4]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [5]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [6]:
#Sorting by Txn DT
train_df = train_df.sort_values("TransactionDT").reset_index(drop=True)
valid_df = valid_df.sort_values("TransactionDT").reset_index(drop=True)

In [7]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [8]:
available_features = [c for c in base_features if c in train_df.columns]

In [9]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [10]:
new_features = ["log_txnamt", "has_identity"]

In [11]:
feature_cols=available_features+new_features

#### Model Training and Result Function

In [12]:
def model_input_data(train_df,valid_df,feature_cols,target_col="isFraud"):

    #Splitting Target vs Predictors
    X_train=train_df[feature_cols].copy()
    X_valid=valid_df[feature_cols].copy()

    y_train=train_df[target_col].copy()
    y_valid=valid_df[target_col].copy()

    #Seperating Categorical and Numerical Columns
    cat_cols=X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols=X_train.select_dtypes(include="number").columns.tolist()

    #Filling Missing Values
    for col in num_cols:
        X_train[col]=X_train[col].fillna(X_train[col].median())
        X_valid[col]=X_valid[col].fillna(X_valid[col].median())

    for col in cat_cols:
        X_train[col]=X_train[col].fillna("Missing").astype(str)
        X_valid[col]=X_valid[col].fillna("Missing").astype(str)
    
    # Typecasting all strings into category
    for col in cat_cols:
        X_train[col]=X_train[col].astype("category")
        X_valid[col]=X_valid[col].astype("category")
    
    return X_train, X_valid, y_train, y_valid, cat_cols

In [13]:
def model_results(train_df,valid_df,feature_cols,model_name="lgbm"):
    X_train, X_valid, y_train, y_valid, cat_cols=model_input_data(train_df,valid_df,feature_cols)
    model=LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=64,subsample=0.8,
                         colsample_bytree=0.8,random_state=42,class_weight="balanced")
    model.fit(X_train, y_train, categorical_feature=cat_cols)

    pred=model.predict_proba(X_valid)[:,1]

    results = {
        "n_features": len(feature_cols),
        "roc_auc": roc_auc_score(y_valid,pred),
        "pr_auc":average_precision_score(y_valid,pred),
        "pred":pred,
        "model":model
    }

    return results

#### Baseline Results

In [14]:
baseline_result = model_results(train_df, valid_df, feature_cols)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026325 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [15]:
print("Baseline ROC-AUC:", round(baseline_result["roc_auc"], 5))
print("Baseline PR-AUC :", round(baseline_result["pr_auc"], 5))

Baseline ROC-AUC: 0.80776
Baseline PR-AUC : 0.23062


#### Adding Velocity Features

```sql
Adding Velocity features:-

- Add a timestamp converted to seconds column 
- For each time window (1H, 24H, 7D) etc:
    compute current time - window time and select all records between those
    do count, sum or average as required

```

In [61]:
df = train_df.copy()

df = df.sort_values("t_s")

In [62]:
df = df.set_index("t_s")

In [67]:
df["test_cnt_1h"] = df["TransactionAmt"].rolling("1h").count()

In [68]:
def add_velocity_features(df, entity_col, windows, time_col="TransactionDT", amt_col="TransactionAmt", prefix=None):

    df = df.copy()
    prefix = prefix or entity_col

    #Coverting timestamp to ns so that pandas recognizes 1h, 24h kind of windows
    df["t_s"] = pd.to_datetime(df["TransactionDT"], unit="s").astype("datetime64[ns]")

    #Sorting dataset in increasing order of timestamp
    df = df.sort_values(["t_s"])

    #Setting index as timestamp so that rolling works as expected
    df = df.set_index("t_s")

    #Creating grouped Entities => this will create multi-index (Entity x timestamp)
    df_g = df.groupby(entity_col, sort=False)

    for w in windows:
        roll=df_g.rolling(w,closed="left")
        
        df[f"{prefix}_cnt_{w}"] = (roll.count().reset_index(level=0, drop=True)) #drop Entity index at level 0
        df[f"{prefix}_sum_{w}"] = (roll.sum().reset_index(level=0, drop=True))

    # Resetting index back to integers => drop timestamp indes
    df = df.reset_index(drop=True)
    df.drop(columns=["t_s"], inplace=True) #drop the new timestamp column as well
    return df

In [71]:
windows=["1h","24h"] #keep standard windows which pandas understands

In [72]:
train_v=add_velocity_feature(train_df,"card1",windows=windows)

In [73]:
valid_v=add_velocity_feature(valid_df,"card1",windows=windows)

In [75]:
vel_cols=[col for col in train_v.columns if col.endswith("_1h") or col.endswith("_24h")]

#### Model Results with Velocity Columns

In [77]:
feature_cols_vel=feature_cols+vel_cols

In [78]:
vel_feature_result = model_results(train_v, valid_v, feature_cols_vel)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025345 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2628
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


#### Comparing the results

In [79]:
df_comp=pd.DataFrame([
    {
        "version":"baseline",
        "n_feature":baseline_result["n_features"],
        "roc_auc":baseline_result["roc_auc"],
        "pr_auc":baseline_result["pr_auc"],
    },
    {
        "version":"vel_feature_result",
        "n_feature":vel_feature_result["n_features"],
        "roc_auc":vel_feature_result["roc_auc"],
        "pr_auc":vel_feature_result["pr_auc"],
    }
])

In [81]:
df_comp.sort_values(["pr_auc", "roc_auc"], ascending=False)

,version,n_feature,roc_auc,pr_auc
0,baseline,16,0.807761,0.230621
1,vel_feature_result,20,0.804145,0.229612


#### End